# Google Trends preprocessing — season-holdout-safe variant

`main.ipynb` fits denoise (`gtrends_denoise.do_denoise`) and detrend
(`gtrends_detrend.do_detrend`) on the **entire** GT history in one shot —
correct for the operational/production file (`google_trends_preprocessed.csv`,
used by `run/run_weekly.sh` etc.), but not safe for a season-holdout
evaluation: the detrend step's ADF test + trend polynomial is fit using data
through today, including the 2025/26 test season and beyond, so even
*pre-season* feature values end up computed with knowledge of the future.

This notebook produces a second, leakage-free file instead:
1. Split the raw long series at the season boundary using **`io.split_train_test`**
   — already in the codebase, already parameterized with exactly this season
   (train ends 2025-10-05, test runs 2025-10-05 to 2026-05-17, with a 20-week
   warm-up overlap the denoiser's rolling window needs), just never wired up.
2. Fit `do_denoise`/`do_detrend` on the **train** slice only.
3. Replay those *frozen* parameters onto the **test** slice via the existing
   `apply_denoise_test`/`apply_detrend_test` — no re-fitting.
4. Concatenate train + test into one full-coverage, leakage-free file:
   `google_trends_preprocessed_season_holdout.csv`.

`main.ipynb` (operational, whole-history fit) is untouched — this is a
separate output for `Season_Holdout_Gridsearch.ipynb` to point `GT_FILE` at.

## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
from lgbm_forecast import gtrends_download, io, gtrends_cluster, gtrends_denoise, gtrends_detrend

# The one date that actually matters -- defined here (not just at the split step) because
# it now also governs the clustering decisions below (Steps 3-5), not just denoise/detrend.
# Matches Season_Holdout_Gridsearch.ipynb's TEST_SEASON_START (ISO week 40, 2025) exactly.
TRUE_TEST_SEASON_START = "2025-10-05"

print("import OK")


import OK


## Steps 1-5 — same as `main.ipynb`, but the OK-vs-CLUSTER and 30%-zero
## recheck decisions are now made on pre-season data only

Loading raw data is unchanged from the operational notebook — reused
verbatim, `RUN_DOWNLOAD`/`RUN_CLUSTER_DOWNLOAD` stay off, assumes the raw
per-keyword CSVs `main.ipynb` already downloaded are present under `data/raw/`.

What *is* different from `main.ipynb`: the two zero-percentage checks below
(`make_zero_df`'s OK-vs-CLUSTER classification, and the recheck that decides
whether a location's combined cluster is still too sparse to use) now compute
their statistics using only data before `TRUE_TEST_SEASON_START`, not the
whole history. These are structural/categorical decisions (which columns
even exist as features) rather than continuous feature values, but the same
"don't let future data influence pre-season decisions" principle applies —
and it does change one outcome: `SI`'s cluster sits right at the 30%
threshold and flips between whole-history (included) and pre-season-only
(excluded) views. The actual retained *values*, once a column/location
passes the check, still span the full history — only the decision of
whether to include them is season-aware.

In [2]:
LOCATIONS = gtrends_download.load_hub_locations()
print(len(LOCATIONS), "locations")


32 locations


In [3]:
google_df = io.load_timeseries_wide(str(RAW_DIR), LOCATIONS)
google_df = google_df.drop(columns=["cough cold & flu"], errors="ignore")
print(google_df.shape)


(21088, 23)


In [4]:
# Decide OK-vs-CLUSTER using pre-season data only (this also makes the
# correlation-based duplicate-removal inside make_zero_df pre-season-only,
# since it reuses the same df). The actual raw values kept later still span
# the full history -- only this classification decision is truncated.
google_df_train = google_df[google_df["date"] < TRUE_TEST_SEASON_START]

zero_df = gtrends_cluster.make_zero_df(google_df_train)
ok_all = gtrends_cluster.make_ok_dict(zero_df)

cluster_all = (
    zero_df[zero_df["action"] == "CLUSTER"]
    .groupby("location")["column"]
    .apply(lambda cols: " + ".join(sorted(cols)))
    .reset_index(name="terms")
)
cluster_all["cluster_id"] = 1
cluster_all = cluster_all[["location", "cluster_id", "terms"]]

google_cluster = io.load_timeseries_wide(str(RAW_DIR), LOCATIONS, only_clusters=True)
print(google_cluster.shape)


action
OK         360
CLUSTER    191
Name: count, dtype: int64
(21088, 3)


In [5]:
# Same fix as above: the 30%-zero recheck decides whether a location's cluster
# is usable at all, using pre-season data only -- e.g. SI sits right on the
# threshold and flips from included (whole-history: 29.4% zero) to excluded
# (pre-season-only: 30.8% zero).
google_cluster_train = google_cluster[google_cluster["date"] < TRUE_TEST_SEASON_START]

cluster_zero_check = (
    google_cluster_train.groupby("location")["cluster_all"]
    .apply(lambda s: (s == 0).mean() * 100)
    .reset_index(name="zero_pct")
)
cluster_zero_check["under_30pct"] = cluster_zero_check["zero_pct"] < 30
location_no_cluster = cluster_zero_check[cluster_zero_check["under_30pct"] == False]["location"].tolist()
print(f"Locations dropped (cluster still >=30% zero as of pre-season data): {location_no_cluster}")

long_ok = gtrends_denoise.make_selected_long(google_df, ok_all)
cluster_dict = {loc: ["cluster_all"] for loc in google_cluster["location"].unique()}
for loc in location_no_cluster:
    cluster_dict.pop(loc, None)
long_cluster = gtrends_denoise.make_selected_long(google_cluster, cluster_dict)

long_all = pd.concat([long_ok, long_cluster], ignore_index=True)
print(f"{long_all.shape[0]} rows, date range {long_all['date'].min()} to {long_all['date'].max()}")
long_all.head()


Locations dropped (cluster still >=30% zero as of pre-season data): ['FR', 'NL', 'SI']
256351 rows, date range 2013-12-29 to 2026-08-09


,date,location,variable,value
0,2013-12-29,AT,cold medicine,254.111014
1,2014-01-05,AT,cold medicine,188.893302
2,2014-01-12,AT,cold medicine,212.712864
3,2014-01-19,AT,cold medicine,291.110805
4,2014-01-26,AT,cold medicine,303.850721


# Step 6 — Season split

`io.split_train_test` defaults already match the season exactly. Two different
notions of "test" are in play here, deliberately:
- **`train_long`**: `2014-10-01` up to (exclusive) `2025-10-05` (ISO week 40,
  2025) — the same boundary `Season_Holdout_Gridsearch.ipynb` uses as
  `TEST_SEASON_START`/`GRIDSEARCH_ANCHOR`.
- **`test_long`** (internal working set only): `2025-05-18` to `2026-05-17` —
  starts 20 weeks *before* `TRUE_TEST_SEASON_START` on purpose, so the
  denoiser's rolling window (`WINDOW=20`) has enough preceding raw values to
  produce a real (non-NaN) smoothed value right at the true season start.
  This buffer never reaches the saved file — `apply_denoise_test`'s
  `keep_from=TRUE_TEST_SEASON_START` below trims it straight back off.

So: the *deliverable* always starts exactly at week 40 (`2025-10-05`),
regardless of this internal 20-week lead-in.

In [6]:
train_long, test_long = io.split_train_test(long_all)
print(f"train_long        : {train_long.shape[0]} rows, {train_long['date'].min()} to {train_long['date'].max()}")
print(f"test_long (buffer): {test_long.shape[0]} rows, {test_long['date'].min()} to {test_long['date'].max()}  "
      f"(includes 20-week warm-up BEFORE {TRUE_TEST_SEASON_START} -- trimmed off before saving)")


train_long        : 223286 rows, 2014-10-05 to 2025-09-28
test_long (buffer): 20617 rows, 2025-05-18 to 2026-05-17  (includes 20-week warm-up BEFORE 2025-10-05 -- trimmed off before saving)


## Step 7 — Denoise: fit on train, replay onto test

`keep_from="2025-10-05"` drops the warm-up rows from the *output* — they were
only needed so `rolling_denoise`'s trailing window has real data to work with
right at the season boundary.

In [7]:
denoised_train, denoise_summary = gtrends_denoise.do_denoise(train_long)
print(denoise_summary["denoised"].value_counts())

denoised_test = gtrends_denoise.apply_denoise_test(test_long, denoise_summary, keep_from=TRUE_TEST_SEASON_START)
print(f"denoised_train: {denoised_train.shape[0]} rows, max date {denoised_train['date'].max()}")
print(f"denoised_test : {denoised_test.shape[0]} rows, "
      f"{denoised_test['date'].min()} to {denoised_test['date'].max()}")
assert str(denoised_test["date"].min().date()) == TRUE_TEST_SEASON_START, \
    "warm-up buffer wasn't fully trimmed -- test data doesn't start at the true season boundary"
print(f"Confirmed: test data starts exactly at {TRUE_TEST_SEASON_START}, no earlier.")


denoised
True     325
False     64
Name: count, dtype: int64
denoised_train: 223286 rows, max date 2025-09-28
denoised_test : 12837 rows, 2025-10-05 00:00:00 to 2026-05-17 00:00:00
Confirmed: test data starts exactly at 2025-10-05, no earlier.


## Step 8 — Detrend: fit on denoised train, replay onto denoised test

`apply_detrend_test` expects the test slice on its own (not concatenated with
train) — it continues the fitted trend polynomial forward from where training
left off, so passing train+test together would double-count the training
rows in that continuation index.

In [8]:
pre_detrended, detrend_summary = gtrends_detrend.do_detrend(denoised_train)
post_detrended = gtrends_detrend.apply_detrend_test(denoised_test, detrend_summary)

detrended_all_clean = pd.concat([pre_detrended, post_detrended], ignore_index=True)
print(f"{detrended_all_clean.shape[0]} rows total  "
      f"(nulls in detrended_value: {detrended_all_clean['detrended_value'].isna().sum()})")

recheck = gtrends_detrend.recheck_stationarity_adf(pre_detrended, column="detrended_value")
recheck["stationarity_status"].value_counts()


236123 rows total  (nulls in detrended_value: 6)


stationarity_status
stationary around constant    389
Name: count, dtype: int64

## Save

In [9]:
gt_wide_clean = detrended_all_clean.pivot_table(
    index=["date", "location"], columns="variable", values="detrended_value", aggfunc="first"
).reset_index()
gt_wide_clean.columns.name = None
gt_wide_clean["date"] = pd.to_datetime(gt_wide_clean["date"])
gt_wide_clean = gt_wide_clean[gt_wide_clean["date"] >= "2014-10-01"].reset_index(drop=True)

out_path = PROCESSED_DIR / "google_trends_preprocessed_season_holdout.csv"
gt_wide_clean.to_csv(out_path, index=False)
print(f"Saved: {out_path}  {gt_wide_clean.shape}  "
      f"({gt_wide_clean['date'].min().date()} to {gt_wide_clean['date'].max().date()})")
gt_wide_clean.head()


Saved: /home/nadillia/Documents/MIGHTE-respicast-jointGBM/google_preprocessing/data/processed/google_trends_preprocessed_season_holdout.csv  (19424, 19)  (2014-10-05 to 2026-05-17)


,date,location,avian influenza,cluster_all,cold medicine,common cold,cough,dyspnea,fever,headache,influenza,influenza vaccine,nasal congestion,nausea,paracetamol,runny nose,sore throat,throat lozenge,virus
0,2014-10-05,AT,NaN,482.479371,266.356805,1604.510721,1704.918857,332.477377,1952.212113,1687.051708,974.777554,NaN,153.102664,1262.527753,788.561006,NaN,914.831202,NaN,3742.921606
1,2014-10-05,BE,NaN,401.333782,NaN,965.129751,955.089433,141.348702,1329.458519,1311.504482,943.477369,NaN,175.584967,1020.880573,804.605966,98.751749,574.071783,0.0,3429.966731
2,2014-10-05,BG,NaN,218.032850,NaN,1106.323278,2791.402049,201.706475,NaN,727.856478,1154.420208,NaN,NaN,607.988012,708.171237,1576.947952,222.122446,NaN,3141.772606
3,2014-10-05,CH,NaN,362.319469,107.380068,1222.042898,1268.146399,272.985519,1643.081738,1694.516108,1200.108293,NaN,157.644820,1460.471255,948.041876,0.000000,589.475989,NaN,3578.876017
4,2014-10-05,CY,NaN,767.632346,NaN,1228.881558,1417.855209,NaN,1593.834341,1656.683825,0.000000,NaN,NaN,0.000000,1.776327,NaN,0.000000,NaN,5315.237239


## Verify the leak is actually gone

Same check that found the problem in the first place: `CZ` / `sore throat`,
around the season boundary. The operational (whole-series-fit) file decided
this series needed a linear detrend — informed by data through 2026-08; the
clean (pre-season-only-fit) file makes that decision blind to anything after
2025-09-28.

In [10]:
CHECK_LOCATION = "CZ"
CHECK_TERM = "sore throat"

clean_check = gt_wide_clean[gt_wide_clean["location"] == CHECK_LOCATION].set_index("date")[CHECK_TERM]
clean_check = clean_check.loc["2025-09-01":"2025-10-12"]

operational = pd.read_csv(PROCESSED_DIR / "google_trends_preprocessed.csv", parse_dates=["date"])
op_check = operational[operational["location"] == CHECK_LOCATION].set_index("date")[CHECK_TERM]
op_check = op_check.loc["2025-09-01":"2025-10-12"]

comparison = pd.DataFrame({"operational (leaky)": op_check, "season_holdout (clean)": clean_check})
comparison["pct_diff"] = 100 * (comparison["operational (leaky)"] - comparison["season_holdout (clean)"]) / comparison["season_holdout (clean)"]
comparison


,operational (leaky),season_holdout (clean),pct_diff
date,,,
2025-09-07,982.812673,522.403201,88.132973
2025-09-14,1226.425265,650.579529,88.512735
2025-09-21,1264.078406,670.030803,88.659745
2025-09-28,1171.107524,620.240258,88.815142
2025-10-05,1120.894392,592.325872,89.236102
2025-10-12,1028.494344,542.170778,89.699332
